In [ ]:
import time
import json
import os
import sys
import math
sys.path.append(os.path.dirname(os.getcwd()))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar,DataLoader

from network.layer import Layer

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=20,adc_last_gap=20)
chip.adc.set_gain_resistor(big_resistance=22e-3,small_resistance=200)
chip.clk_manager.set_cyc(delay1=20,delay2=50)
chip.add_compiler("../compiler/code/")
# chip.compensation.initop("./modules/data/")

In [ ]:
point_read = np.zeros((256,256))
sum_read = np.zeros((1,256))
num = 40
for i in range(num):
    voltage_base = chip.read_point3(0,256,0,256,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point3(0,256,0,256,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    print(np.max(voltage))
    resistence = chip.voltage_to_resistance(voltage=voltage-voltage_base)

    point_read+=resistence

    voltage_base=chip.compute(crossbar=np.ones((256,256)),read_voltage=0,tg=5,gain=3,from_row=True,out_type=0)
    voltage=chip.compute(crossbar=np.ones((256,256)),read_voltage=0.1,tg=5,gain=3,from_row=True,out_type=0)
    # print(voltage.shape)
    print(np.max(voltage))
    resistence = chip.voltage_to_resistance(voltage = voltage-voltage_base)

    sum_read+=resistence

point_read=point_read/num
sum_read=sum_read/num

In [ ]:
point_read2=point_read*1000
sum_read2=sum_read*1000

for row in range(256):
    for col in range(256):
        point_read2[row,col] -= (255-row)*0.12 if col%2==0 else row*0.12

In [ ]:
r_out = np.zeros((256))
for col in range(256):
    # 256列
    
    # 每列遍历256行
    left = 10
    right = 30
    mid = 0
    while right-left>0.01:
        mid = (left+right)/2
        tmp = point_read2[0,col]-mid
        
        for row in range(1,256):
            # print(tmp)
            tmp = (tmp+0.12)*(point_read2[row,col]-mid)/(tmp+0.12+point_read2[row,col]-mid)
        if tmp>sum_read2[0,col]-mid:
            # r_out偏大
            right=mid
        else:
            left=mid
        # print(tmp+mid,sum_read2[0,col])
    r_out[col]=mid
    # print(mid)
        

In [ ]:
r_out[r_out<15]=18
r_out[r_out>25]=18
plt.plot(r_out)
plt.show()
np.save("../chip_data/chip8_/col_r_out.npy",r_out)
# np.save("../chip_data/chip8_/col_r_wire.npy",R_wire_mean)

In [ ]:
R_out_mean,R_wire_mean = chip.compensation.calculate_r_out_r_wire(chip=chip,col=True,nums=30,sub_base=True)

In [ ]:
np.save("../chip_data/chip8_/col_r_out.npy",R_out_mean)
np.save("../chip_data/chip8_/col_r_wire.npy",R_wire_mean)

In [ ]:
col_r_out = "../chip_data/chip_1_4/col_r_out.npy"
col_r_wire = "../chip_data/chip_1_4/col_r_wire.npy"

col_r_out = "../chip_data/chip8_/col_r_out.npy"
col_r_wire = "../chip_data/chip8_/col_r_wire.npy"

R_out_mean2 = np.load(col_r_out)
R_wire_mean2 = np.load(col_r_wire)
# np.save(r_wire_path,R_wire_mean)
R_out_mean2 = np.nan_to_num(R_out_mean2, nan=18)
R_wire_mean2 = np.nan_to_num(R_wire_mean2, nan=0.12)
R_out_mean2[R_out_mean2>30]=17
R_out_mean2[R_out_mean2<10]=17
R_wire_mean2[R_wire_mean2>1]=0.12
R_wire_mean2[R_wire_mean2<0]=0.12
plt.figure(figsize=(12,4))
plt.subplot(1, 2, 1)
plt.plot(R_out_mean2[:])
plt.title("R_out")
plt.xlabel("col")
plt.ylabel("Ω")

plt.subplot(1, 2, 2)
plt.plot(R_wire_mean2[:])
plt.title("R_wire")
plt.xlabel("col")
plt.ylabel("Ω")


plt.tight_layout()
plt.show()

In [ ]:
R_out_mean2 = np.load(col_r_out)
for k in R_wire_mean2:
    print(k)

# 1.计算R_out和R_wire

In [ ]:
def get_A_B_C(pos,num,from_row,sub_base=False):
    need_read = np.zeros((256,256),dtype=bool)
    weight_pos = np.ix_([i for i in range(pos[0],pos[1])], [num]) if from_row else np.ix_([num],[i for i in range(pos[0],pos[1])])
    need_read[weight_pos] = True
    gain = 1 if (pos[1]-pos[0])<15 else 3
    v = 0.2 if (pos[2]-pos[0])<15 else 0.1
    if sub_base:
        voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=gain,from_row=from_row,out_type=0)
    voltage = chip.compute(crossbar=need_read,read_voltage=v,tg=5,gain=gain,from_row=from_row,out_type=0)
    print("最大电压",np.max(voltage))
    if sub_base:
        voltage -= voltage_base
    a = chip.voltage_to_resistance(voltage = voltage)

    # 如果第二段开的行太少,3挡增益不太够
    need_read[:] = False
    weight_pos = np.ix_([i for i in range(pos[1],pos[2])], [num]) if from_row else np.ix_([num],[i for i in range(pos[1],pos[2])])
    need_read[weight_pos] = True
    gain = 1 if (pos[2]-pos[1])<15 else 3
    v = 0.2 if (pos[2]-pos[0])<15 else 0.1
    if sub_base:
        voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=gain,from_row=from_row,out_type=0)
    voltage = chip.compute(crossbar=need_read,read_voltage=v,tg=5,gain=gain,from_row=from_row,out_type=0)
    print("最大电压",np.max(voltage))
    if sub_base:
        voltage -= voltage_base
    b = chip.voltage_to_resistance(voltage = voltage)

    need_read[:] = False
    weight_pos = np.ix_([i for i in range(pos[0],pos[2])], [num]) if from_row else np.ix_([num],[i for i in range(pos[0],pos[2])])
    need_read[weight_pos] = True
    gain = 1 if (pos[2]-pos[0])<15 else 3
    v = 0.2 if (pos[2]-pos[0])<15 else 0.1
    if sub_base:
            voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=gain,from_row=from_row,out_type=0)
    voltage = chip.compute(crossbar=need_read,read_voltage=v,tg=5,gain=gain,from_row=from_row,out_type=0)
    print("最大电压",np.max(voltage))
    if sub_base:
        voltage -= voltage_base
    c = chip.voltage_to_resistance(voltage = voltage)

    if from_row:
        return a[0,num],b[0,num],c[0,num]
    else:
        return a[num,0],b[num,0],c[num,0]
    
def get_out(pos,num,from_row,sub_base=True):
    a,b,c = get_A_B_C(pos=pos,num=num,from_row=from_row,sub_base=sub_base)

    base1 = ((c-a)/(c-b))**0.5
    base2 = ((c-b)/(c-a))**0.5

    if base1>1:
        R2 = (b-c)*(1+base1)
        R = b-R2
    elif base2>1:
        R1 = (a-c)*(1+base2)
        R = a-R1

    return R

In [ ]:
# R=get_out((0,6,56),10,from_row=False)
# print(R)

# R=get_out((200,252,256),11,from_row=False)
# print(R)

# R=get_out((0,6,56),11,from_row=True)-get_out((200,250,256),11,from_row=True)
# print(R)

# R=get_out((200,250,256),200,from_row=True)
# print(R)

R=(get_out((200,210,256),11,from_row=True)-get_out((0,10,56),11,from_row=True))/200*1000
print(R)

In [ ]:
nums = 50
# ----------------------------------------------------------------列的
R_out = np.zeros((256,nums))
R_wire = np.zeros((256,nums))
for col in range(256):
    print("列",col)
    for j in range(nums):
        if col%2==0:
            R_out[col,j] = get_out((200,240,256),col,from_row=True)
            R_wire[col,j] = get_out((0,40,56),col,from_row=True)
        else:
            R_out[col,j] = get_out((0,16,56),col,from_row=True)
            R_wire[col,j] = get_out((200,216,256),col,from_row=True)

nums = 50
# ----------------------------------------------------------------行的
# R_out = np.zeros((256,nums))
# R_wire = np.zeros((256,nums))
# for row in range(256):
#     print("行",row)
#     for j in range(nums):
#         if row%2==0:
#             R_out[row,j] = get_out((0,6,56),row,from_row=False)
#             R_wire[row,j] = get_out((200,206,256),row,from_row=False)
#         else:
#             R_out[row,j] = get_out((200,250,256),row,from_row=False)
#             R_wire[row,j] = get_out((0,50,56),row,from_row=False)


In [ ]:
r_out_path = "../data/r_out_r_wire/chip_1_4_col_r_out_0_50_56_no_sub_base.npy"
r_wire_path = "../data/r_out_r_wire/chip_1_4_col_r_wire_0_50_56_no_sub_base.npy"
# r_out_path = "../data/r_out_r_wire/chip_1_4_col_r_out_0_50_56_times=2.npy"
# r_wire_path = "../data/r_out_r_wire/chip_1_4_col_r_out_0_50_56_times=2.npy"
R_out_mean = np.mean(R_out,axis=1)*1000
# np.save(r_out_path,R_out_mean)

R_out_read = np.load(r_out_path)
R_wire_mean = (np.mean(R_wire,axis=1)*1000-R_out_read)/200
# np.save(r_wire_path,R_wire_mean)

plt.figure(figsize=(12,4))
plt.subplot(1, 2, 1)
plt.plot(R_out_mean)
plt.title("R_out")
plt.xlabel("col")
plt.ylabel("Ω")

plt.subplot(1, 2, 2)
plt.plot(R_wire_mean)
plt.title("R_wire")
plt.xlabel("col")
plt.ylabel("Ω")


plt.tight_layout()
plt.show()

In [ ]:
r_out = np.load("../data/r_out_r_wire/chip_1_4_col_r_out_0_50_56_times=2.npy")
r_w = np.load("../data/r_out_r_wire/chip_1_4_col_r_wire_0_50_56_times=2.npy")
# r_out = np.load("../data/r_out_r_wire/chip_1_4_col_r_out_0_50_56_no_sub_base.npy")
# r_w = np.load("../data/r_out_r_wire/chip_1_4_col_r_wire_0_50_56_no_sub_base.npy")

plt.figure(figsize=(12,4))
plt.subplot(1, 2, 1)
plt.plot(r_out)
plt.title("R_out")
plt.xlabel("col")
plt.ylabel("Ω")

plt.subplot(1, 2, 2)
plt.plot(r_w)
plt.title("R_wire")
plt.xlabel("col")
plt.ylabel("Ω")


plt.tight_layout()
plt.show()

# 2.计算实际电导值

In [ ]:
r_out = np.load("../data/r_out_r_wire/chip_1_4_col_r_out_0_50_56_times=2.npy")
r_w = np.load("../data/r_out_r_wire/chip_1_4_col_r_wire_0_50_56_times=2.npy")

# r_out = np.load("../data/r_out_r_wire/chip_1_4_col_r_out_0_50_56_no_sub_base.npy")
# r_w = np.load("../data/r_out_r_wire/chip_1_4_col_r_wire_0_50_56_no_sub_base.npy")


In [ ]:
row = [i for i in range(0,256)]
col = 154
weight_pos = np.ix_(row, [col])
need_read = np.zeros((256,256),dtype=bool)
need_read[weight_pos] = True

voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
# need_write = cond<400 & need_read
# chip.write_point2(crossbar=need_write,write_voltage=5,tg=2,pulse_width=1000e-6,set_device=True)
resistance_real = chip.voltage_to_resistance(voltage=voltage-voltage_base)[weight_pos]*1e3 - r_out[col]


# 每一行有不同的列电阻
for i in row:
    r_w_sum = (255-i)*r_w[col] if col%2==0 else i*r_w[col]
    resistance_real[i,0] -= r_w_sum

real_cond = 1/resistance_real*1e6
print(np.mean(real_cond))
plt.plot(real_cond.flatten())
plt.title(f"col={col} real_cond")
plt.ylabel("uS")
plt.xlabel("row")
plt.show()

cond0_ans = [np.sum(real_cond[0:i+1]) for i in range(0,256)]

In [ ]:
cond1_ans = []
cond2_ans = []
cond3_ans = []
for k in range(0,256):
    print("k=",k)
    row = [i for i in range(0,k+1)]

    # selected_elements = random.sample(my_list, 50)
    weight_pos = np.ix_(row, [col])
    need_read = np.zeros((256,256),dtype=bool)
    need_read[weight_pos] = True


    row_min,row_max = np.min(row),np.max(row)
    r_w_sum = (255-row_max)*r_w[col] if col%2==0 else row_min*r_w[col]
    # if len(row)<40:
    #     r_w_sum = 0


    gain = 3#1 if len(row)<15 else 3
    voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=gain,from_row=True,out_type=0)
    voltage = chip.compute(crossbar=need_read,read_voltage=0.1,tg=5,gain=gain,from_row=True,out_type=0)
    resistance = chip.voltage_to_resistance(voltage=voltage-voltage_base)[0,col]*1e3
    cond2 = 1/resistance*1e6
    cond2_ans.append(cond2)

    n = len(row)
    a = (n*(n-1)/2)
    b = n+n*(n+1)*(n-1)/6*(r_w[col]+r_w[col]**2+r_w[col]**3)/np.mean(real_cond)
    # print(a,b,a/b,n/b)
    resistance = resistance - r_out[col] - r_w_sum
    # if gain == 1:
    #     resistance -=  r_out[col]
    cond1 = 1/resistance*1e6
    cond1_ans.append(cond1)

    resistance -= a/b*r_w[col]
    cond3 = 1/resistance*1e6*0.925
    cond3_ans.append(cond3)

In [ ]:
num=256
plt.plot(cond0_ans[:num],label = "real")
plt.plot(cond3_ans[:num],label = "Compensation2")
plt.plot(cond1_ans[:num],label = "Compensation only r_out")
plt.plot(cond2_ans[:num],label ="No Compensation")

print(cond0_ans)
print(cond1_ans)
print(cond2_ans)
print(cond3_ans)
print(len(cond0_ans))
print(len(cond1_ans))
print(len(cond2_ans))
plt.title(f"col={col}")
plt.ylabel("uS")
plt.xlabel("try times")
plt.legend()
plt.show()

plt.plot(np.array(cond3_ans)/np.array(cond0_ans),label = "Compensation2/real")
plt.plot(np.array(cond1_ans)/np.array(cond0_ans),label = "Compensation only r_out/real")
plt.plot(np.array(cond2_ans)/np.array(cond0_ans),label = "No Compensation/real")

print(np.array(cond3_ans)/np.array(cond0_ans))
plt.title(f"col={col}")
# plt.ylabel("uS")
plt.xlabel("try times")
plt.legend()
plt.show()

# 3.由真实值重构最后的累加值

In [ ]:
r_out = np.load("../data/r_out_r_wire/chip_1_4_col_r_out_0_50_56_times=2.npy")
r_w = np.load("../data/r_out_r_wire/chip_1_4_col_r_wire_0_50_56_times=2.npy")

In [ ]:
row = [i for i in range(0,256)]
col = 155
weight_pos = np.ix_(row, [col])
need_read = np.zeros((256,256),dtype=bool)
need_read[weight_pos] = True

voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
# need_write = cond<400 & need_read
# chip.write_point2(crossbar=need_write,write_voltage=5,tg=2,pulse_width=1000e-6,set_device=True)
resistance_real = chip.voltage_to_resistance(voltage=voltage-voltage_base)[weight_pos]*1e3 - r_out[col]


# 每一行有不同的列电阻
for i in row:
    r_w_sum = (255-i)*r_w[col] if col%2==0 else i*r_w[col]
    resistance_real[i,0] -= r_w_sum

real_cond = 1/resistance_real*1e6
print(np.mean(real_cond))
plt.plot(real_cond.flatten())
plt.title(f"col={col} real_cond")
plt.ylabel("uS")
plt.xlabel("row")
plt.show()

cond0_ans = [np.sum(real_cond[0:i+1]) for i in range(0,256)]

In [ ]:
cond1_ans = []
cond2_ans = []
cond3_ans = []
for k in range(0,256):
    print("k=",k)
    row = [i for i in range(0,k+1)]

    # selected_elements = random.sample(my_list, 50)
    weight_pos = np.ix_(row, [col])
    need_read = np.zeros((256,256),dtype=bool)
    need_read[weight_pos] = True


    row_min,row_max = np.min(row),np.max(row)
    r_w_sum = (255-row_max)*r_w[col] if col%2==0 else row_min*r_w[col]
    # if len(row)<40:
    #     r_w_sum = 0


    gain = 3#1 if len(row)<15 else 3
    voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=gain,from_row=True,out_type=0)
    voltage = chip.compute(crossbar=need_read,read_voltage=0.1,tg=5,gain=gain,from_row=True,out_type=0)
    resistance = chip.voltage_to_resistance(voltage=voltage-voltage_base)[0,col]*1e3
    cond2 = 1/resistance*1e6
    cond2_ans.append(cond2)

    n = len(row)
    a = (n*(n-1)/2)
    b = n+n*(n+1)*(n-1)/6*(r_w[col]+r_w[col]**2+r_w[col]**3)/np.mean(real_cond)
    # print(a,b,a/b,n/b)
    resistance = resistance - r_out[col] - r_w_sum
    # if gain == 1:
    #     resistance -=  r_out[col]
    cond1 = 1/resistance*1e6
    cond1_ans.append(cond1)

    resistance -= a/b*r_w[col]
    cond3 = 1/resistance*1e6*0.925
    cond3_ans.append(cond3)

In [ ]:
rw = r_w[col]
real_r_sum = np.zeros((256))
for i in range(256):
    if i==0:
        real_r_sum[i] = resistance_real[i,0]
    else:
        real_r_sum[i] = (real_r_sum[i-1]+rw)*resistance_real[i,0]/((real_r_sum[i-1]+rw)+resistance_real[i,0])

for i in range(256):
    r_w_sum = (255-i)*rw if col%2==0 else i*rw
    real_r_sum[i] += r_w_sum + r_out[i]

real_c_sum = 1/real_r_sum*1e6

num=256
plt.plot(cond0_ans[:num],label = "real")
plt.plot(cond2_ans[:num],label ="No Compensation")
plt.plot(real_c_sum[:num],label = "real-->No Compensation")

plt.title(f"col={col}")
plt.ylabel("uS")
plt.xlabel("row nums")
plt.legend()
plt.show()

In [ ]:
rw = r_w[col]
real_r_sum = np.zeros((256))
for i in range(256):
    if i==0:
        real_r_sum[i] = resistance_real[i,0]
    else:
        real_r_sum[i] = (real_r_sum[i-1]+rw)*resistance_real[i,0]/((real_r_sum[i-1]+rw)+resistance_real[i,0])

for i in range(256):
    r_w_sum = (255-i)*rw if col%2==0 else 0
    real_r_sum[i] += r_w_sum + r_out[i]

real_c_sum = 1/real_r_sum*1e6


num=256
plt.plot(cond0_ans[:num],label = "real")
plt.plot(cond2_ans[:num],label ="No Compensation")
plt.plot(real_c_sum[:num],label = "real-->No Compensation")

plt.title(f"col={col}")
plt.ylabel("uS")
plt.xlabel("row nums")
plt.legend()
plt.show()

# 3. 其他值

In [ ]:
col=150
row = [4]

# selected_elements = random.sample(my_list, 50)
weight_pos = np.ix_(row, [col])
need_read = np.zeros((256,256),dtype=bool)
need_read[weight_pos] = True





chip.set_cim_reset()
voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
resistance = chip.voltage_to_resistance(voltage=voltage)[weight_pos]*1e3
print("电压",np.max(voltage))
print(1111,resistance)
resistance = resistance- - r_out[col]
for i in row:
    r_w_sum = (255-i)*r_w[col] if col%2==0 else i*r_w[col]
    resistance[0,0] -= r_w_sum
    print(r_w_sum,r_out[col])
real_cond = 1/resistance*1e6
print(real_cond)





row_min,row_max = np.min(row),np.max(row)
r_w_sum = (255-row_max)*r_w[col] if col%2==0 else row_min*r_w[col]

gain = 1 if len(row)<15 else 3
voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=gain,from_row=True,out_type=0)
chip.set_cim_reset()
voltage = chip.compute(crossbar=need_read,read_voltage=0.1,tg=5,gain=gain,from_row=True,out_type=0)
print("电压",np.max(voltage))
resistance = chip.voltage_to_resistance(voltage=voltage)[0,col]*1e3
print(1111,resistance)
resistance_real = resistance- - r_out[col] - r_w_sum
cond1 = 1/resistance_real*1e6

print(cond1,r_out[col],r_w_sum)



In [ ]:
import random

cond0_ans = []
cond1_ans = []
cond2_ans = []
cond3_ans = []
for k in range(0,40):
    r_w = np.load("../data/r_out_r_wire/chip_1_4_col_r_wire_0_50_56_times=2.npy")
    print("k=",k)
    row = [i for i in range(0,100)]

    # row = random.sample(row, 50)
    print(len(row))
    weight_pos = np.ix_(row, [col])
    need_read = np.zeros((256,256),dtype=bool)
    need_read[weight_pos] = True

    # voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    # voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    # cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
    # resistance = chip.voltage_to_resistance(voltage=voltage-voltage_base)[weight_pos]*1e3 - r_out[col]
    # real_cond = 1/resistance*1e6
    # cond0_ans.append(np.sum(real_cond))

    # # 每一行有不同的列电阻
    # for i,v in enumerate(row):
    #     r_w_sum = (255-v)*r_w[col] if col%2==0 else v*r_w[col]
    #     resistance[i,0] -= r_w_sum


    row_min,row_max = np.min(row),np.max(row)
    r_w_sum = (255-row_max)*r_w[col] if col%2==0 else row_min*r_w[col]


    r_w = r_w*200/100
    gain = 3#1 if len(row)<15 else 3
    voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=gain,from_row=True,out_type=0)
    voltage = chip.compute(crossbar=need_read,read_voltage=0.1,tg=5,gain=gain,from_row=True,out_type=0)
    resistance = chip.voltage_to_resistance(voltage=voltage-voltage_base)[0,col]*1e3
    cond2 = 1/resistance*1e6
    cond2_ans.append(cond2)

    n = len(row)
    a = (n*(n-1)/2)
    b = n+n*(n+1)*(n-1)/6*(r_w[col]+r_w[col]**2+r_w[col]**3)/np.mean(real_cond)
    print(a,b,a/b,n/b)
    resistance = resistance - r_out[col] - r_w_sum
    cond1 = 1/resistance*1e6
    cond1_ans.append(cond1)

    resistance -= a/b*r_w[col]
    cond3 = 1/resistance*1e6*0.925#*b/n
    cond3_ans.append(cond3)

In [ ]:
print(resistance_real.shape)

In [ ]:
cond1_ans = []
cond2_ans = []
cond3_ans = []
for k in range(0,256):
    print("k=",k)
    row = [i for i in range(0,k+1)]

    # selected_elements = random.sample(my_list, 50)
    weight_pos = np.ix_(row, [col])
    need_read = np.zeros((256,256),dtype=bool)
    need_read[weight_pos] = True


    row_min,row_max = np.min(row),np.max(row)
    r_w_sum = (255-row_max)*r_w[col] if col%2==0 else row_min*r_w[col]
    # if len(row)<40:
    #     r_w_sum = 0


    gain = 3#1 if len(row)<15 else 3
    voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=gain,from_row=True,out_type=0)
    voltage = chip.compute(crossbar=need_read,read_voltage=0.1,tg=5,gain=gain,from_row=True,out_type=0)
    resistance = chip.voltage_to_resistance(voltage=voltage-voltage_base)[0,col]*1e3
    cond2 = 1/resistance*1e6
    cond2_ans.append(cond2)

    n = len(row)
    a = (n*(n-1)/2)
    b = n+n*(n+1)*(n-1)/6*(r_w[col]+r_w[col]**2+r_w[col]**3)/np.mean(real_cond)
    # print(a,b,a/b,n/b)
    resistance = resistance - r_out[col] - r_w_sum
    # if gain == 1:
    #     resistance -=  r_out[col]
    cond1 = 1/resistance*1e6
    cond1_ans.append(cond1)

    resistance -= a/b*r_w[col]
    cond3 = 1/resistance*1e6*0.925
    cond3_ans.append(cond3)

In [ ]:
resistance = 1/np.array(cond2_ans)*1e6

# 实际每个单元的电阻，已经减去col的线阻和r_out
real_r = 1/real_cond[i]*1e6
r_col = R_wire[col]

real_r_cal = []
for i in range(256):
    # 每次迭代时都要用到v和电流
    v_real = 0.1
    current = v_real/resistance[i]
    r_now = resistance[i]
    for k in range(0,k+1):
        
        if k==0:
            row_min,row_max = 0,i
            r_w_sum = (255-row_max)*r_w[col] if col%2==0 else row_min*r_w[col]
            r = r_now - r_out[col] - r_w_sum
            real_r_cal.append(r)
            # 实际分压为
            v_real = current*r
            current -= v_real/real_r[k]
        else:
            r = 


In [ ]:
good_device = np.load("../data/good_point_100_100_1_4.npy")

In [ ]:

weight_pos = np.ix_([i for i in range(256)], [i for i in range(256)])


voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond_add = chip.voltage_to_cond(voltage=voltage)[weight_pos]
cond_add = np.sum(cond_add,axis=0)

voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=3,from_row=True,out_type=0)
voltage = chip.compute(crossbar=need_read,read_voltage=0.1,tg=5,gain=3,from_row=True,out_type=0)
cond_sum = chip.voltage_to_cond(voltage=voltage)
print(np.max(voltage))
plt.plot(cond_add,label="add")
plt.plot(cond_sum[0],label="sum")
plt.legend()
plt.show()

plt.plot((cond_add-cond_sum[0])/cond_add,label="sub")
plt.legend()
plt.show()

In [ ]:
change256 = np.zeros((100,3))
for i in range(100):
    print(i)
    col = good_device[1][-6]

    # need_read = np.zeros((256,256),dtype=bool)
    # weight_pos = np.ix_([i for i in range(0,i+1)],[col])
    # need_read[weight_pos] = True

    need_read = np.zeros((256,256),dtype=bool)
    weight_pos = np.ix_(good_device[0][:i],[col])
    need_read[weight_pos] = True

    wire_r = np.ones((i+1,1))
    for k in range(0,i+1):
        wire_r[k,0]*=r_w[col]*1e-3*k


    chip.set_cim_reset()
    voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    # print(np.max(voltage))
    real_r = chip.voltage_to_resistance(voltage = voltage-voltage_base)[weight_pos]# - r_out[col]*1e-3 - wire_r
    real_cond = np.sum(1/real_r*1e3)
    # print(1/real_cond*1e6)
    # print(np.sum(chip.voltage_to_cond(voltage = voltage-voltage_base)[weight_pos]))

    chip.set_cim_reset()
    voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=3,from_row=True,out_type=0)
    voltage = chip.compute(crossbar=need_read,read_voltage=0.1,tg=5,gain=3,from_row=True,out_type=0)
    # print(np.max(voltage))
    
    real_r = chip.voltage_to_resistance(voltage = voltage-voltage_base)[0,col] - r_out[col]*1e-3 - r_w[col]*i**0.875*1e-3
    if i>40:
        real_r-=(r_w[col]*1e-3)*(i**0.7)
    # print(real_r*1e3)
    cond = 1/real_r*1e3


    chip.set_cim_reset()
    voltage_base = chip.compute(crossbar=need_read,read_voltage=0,tg=5,gain=3,from_row=True,out_type=0)
    voltage = chip.compute(crossbar=need_read,read_voltage=0.1,tg=5,gain=3,from_row=True,out_type=0)
    # print(np.max(voltage))
    real_r = chip.voltage_to_resistance(voltage = voltage-voltage_base)[0,col] #- r_out[col]*1e-3
    # print(real_r*1e3)

    cond2 = 1/real_r*1e3


    change256[i,0]=real_cond
    change256[i,1]=cond
    change256[i,2]=cond2

In [ ]:
plt.plot(change256[:,0],label = "real")
plt.plot(change256[:,1],label = "Compensation")
plt.plot(change256[:,2],label ="No Compensation")
ans = change256[:,0]-change256[:,1]
print(ans[:50])
# plt.title("")
plt.ylabel("uS")
plt.xlabel("row nums")
plt.legend()
plt.show()

In [ ]:
need_read = np.zeros((256,256),dtype=bool)
weight_pos = np.ix_([i for i in range(90,92)],[100])
need_read[weight_pos] = True

voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[weight_pos]
voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[weight_pos]
print(np.max(voltage))
print(chip.voltage_to_cond(voltage=voltage-voltage_base))

voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=3,from_row=True,out_type=0)[weight_pos]
voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=3,from_row=True,out_type=0)[weight_pos]
print(np.max(voltage))
print(chip.voltage_to_cond(voltage=voltage-voltage_base))
# print(voltage[weight_pos])
# real_r = chip.voltage_to_resistance(voltage = voltage)[weight_pos] - r_out[col]*1e-3
# real_cond = 1/real_r*1e3
# print(real_cond)